# M16 · Dimensionality Reduction — Toy Example, Step by Tiny Step

**Companion to lesson M16.** Build **PCA** on correlated engagement-style features: see how a few components capture most of the variance, read the loadings, and note the honest caveat.

## Step 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)

def log(label, value):
    print(f"[{label}] {value}")

log("setup", "tools ready — seed fixed to 0")

## Step 1 · Correlated features → PCA explained variance

We make 3 features that are basically the same signal (clicks/views/dwell all track engagement) plus 1 independent noise feature. PCA finds new axes ordered by how much variance each explains.

In [ ]:
from sklearn.decomposition import PCA
engagement = np.random.normal(0, 1, (200, 1))                 # one hidden 'engagement' signal
X = np.hstack([engagement + 0.1*np.random.normal(size=(200,1)) for _ in range(3)]  # clicks, views, dwell
               + [np.random.normal(0, 1, (200, 1))])                                # independent noise
pca = PCA().fit(X)
cev = np.cumsum(pca.explained_variance_ratio_)
log("explained variance ratio", np.round(pca.explained_variance_ratio_, 3).tolist())
log("cumulative explained variance", np.round(cev, 3).tolist())
assert np.all(np.diff(cev) >= -1e-9)                         # cumulative variance is non-decreasing
n_comp = int(np.searchsorted(cev, 0.90) + 1)                 # components to reach 90%
log("components to reach 90% variance", n_comp)
assert cev[n_comp - 1] >= 0.90

plt.plot(range(1, len(cev)+1), cev, "-o"); plt.axhline(0.90, ls="--", color="red")
plt.title("cumulative explained variance"); plt.xlabel("# components"); plt.ylabel("fraction of variance"); plt.show()

▶ What you'll see: PC1 alone captures most variance (the 3 correlated features), reaching ~90% in 1–2 PCs.

## Step 2 · Loadings — what each component is made of

A component's **loadings** say which original features it mixes. PC1 should load heavily on the three correlated engagement features and little on the noise feature.

In [ ]:
pc1 = pca.components_[0]
log("PC1 loadings [click, view, dwell, noise]", np.round(pc1, 2).tolist())
assert abs(pc1[3]) < max(abs(pc1[:3]))                       # noise feature loads LESS than the engagement trio

plt.bar(["click","view","dwell","noise"], pc1)
plt.title("PC1 loadings (general engagement direction)"); plt.ylabel("weight"); plt.show()

▶ What you'll see: large loadings on click/view/dwell, near-zero on noise.

## Recap

- PCA rotates to axes ordered by variance; a few PCs often capture most of it.
- **Loadings** interpret a component (here PC1 = general engagement).
- Caveat: low-dimensional blobs are **visualization hypotheses**, not labels.